# Практика Prompt Engineering

В этой лабораторной вы попрактикуетесь в технике написания промптов для LLM.

А именно, вы попробуете заставить LLM отвечать правильно, используя техники и структуру промпта, о которых было рассказано в лекции.

In [1]:
from openai import OpenAI

In [2]:
API_KEY="ollama"
BASE_URL="http://localhost:11434/v1"
MODEL_NAME="gemma3:4b"

In [3]:
# Создание клиента
client = OpenAI(api_key=API_KEY, base_url=BASE_URL)

In [4]:
def generate_answer(system_prompt: str, user_prompt: str):
    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ])

    answer = response.choices[0].message.content
    return answer

## Задача 1

В этом задании вам необходимо настроить системную инструкцию для LLM таким образом, чтобы она выдавала ответ в нужном формате. 

LLM в ответ на новость должна выдать один из следующих классов:
- Политика
- Спорт
- Экономика
- Технологии

Примените одну из техник, о которых говорилось в лекции. 

Подумайте, что должно быть в системной инструкции, чтобы LLM точно выдала нужный ответ?

In [5]:
news_texts = [
    "Премьер-министр Великобритании объявил о новых экологических нормах для борьбы с изменением климата.",
    "Команда Реал Мадрид выиграла Лигу чемпионов.",
    "Apple выпустила новый iPhone с улучшенной камерой.",
    "Цены на нефть упали на 5% из-за снижения мирового спроса.",
    "Европейский парламент принял новый климатический закон.",
    "Теннисист Новак Джокович выиграл Открытый чемпионат Австралии.",
    "Google представил ИИ для генерации изображений."
]

true_labels = ["Политика", "Спорт", "Технологии", "Экономика", "Политика", "Спорт", "Технологии"]

Добавил настройку уровня сложности проверки.

В лёгком уровне проверяется, что нужный класс находится в сгенерированном тексте.

    
В сложном уровне проверяется, что LLM выдаёт ТОЛЬКО нужный класс.


Попробуйте решить задачу на обоих уровнях.

    
*Экспериментируйте =)*

In [16]:
DIFFICULTY_LEVEL = 1 # уровень сложности: 0 - легкий, 1 - сложный

In [17]:
def check_classifier(current_answer: str, target_answer: str):
    if DIFFICULTY_LEVEL == 0:
        return target_answer.lower() in current_answer.lower()
    else:
        return target_answer.lower() == current_answer.lower()

Для удобства отладки выводим результаты в консоль. На основе них вы сможете понять, где у LLM могут быть проблемы.

In [23]:
system_prompt = "Ты классификатор . Твоя задача принимать в себя текст и возвращать класс текста . Ответ должен состоять из 1 слова без пробелов , запятых , без поянений и главное без переноса строки и т д . ТОЛЬКО СЛОВО . Список классов : Политика,Спорт,Технологии,Экономика. Классы могут повторяться , выбери 1 слово которое подходит лучше всего. Дам контекст если (цены выросли/снизились , это экономика ), если ( парламент/новый закон/решение министра , это политика)" # напишите системный промпт, который загайдит LLM классифицировать новости на нужные классы

correct_answer = 0

for ind, news in enumerate(news_texts):
    answer = generate_answer(system_prompt=system_prompt, user_prompt=news)

    result = check_classifier(answer, true_labels[ind])

    print()
    print("=" * 20)
    print(f"Generate answer: {answer}")
    print(f"Real answer: {true_labels[ind]}")
    if result:
        correct_answer += result
        print("CORRECTLY")
    else:
     print("WRONG")


Generate answer: Политика

Real answer: Политика
WRONG

Generate answer: Спорт
Real answer: Спорт
CORRECTLY

Generate answer: Технологии
Real answer: Технологии
CORRECTLY

Generate answer: Экономика
Real answer: Экономика
CORRECTLY

Generate answer: Политика

Real answer: Политика
WRONG

Generate answer: Спорт

Real answer: Спорт
WRONG

Generate answer: Технологии
Real answer: Технологии
CORRECTLY


In [21]:
# Доля вернух ответов
print(f"Accuracy = {round(correct_answer / len(true_labels), 2)}")

Accuracy = 0.43


# Задание 2
#### Chain of Thought (Цепочка рассуждений)

В этом задании мы вспомним технику Chain of Thought, а также на практике увидим, как такой подход заставляет модель рассуждать перед выдачей ответа.

Ваша задача - настроить системный промпт таким образом, чтобы LLM начала рассуждать, перед тем как дать ответ.

Также небольшое усложнение (опционально)

Попробуйте добиться, чтобы модель всегда выдавала ответ в формате:

```
Шаг 1: ...
Шаг 2: ...
...
Ответ: ...
```

Для этого добавьте в системный промпт требование строго придерживаться формата.

In [24]:
tasks = [
    "У Маши было 5 яблок. Она отдала 2 яблока Пете, а потом нашла ещё 3. Сколько яблок у Маши теперь?",
    "Если все люди смертны, а Сократ — человек, то смертен ли Сократ?",
    "В магазине скидка 20% на товар стоимостью 1500 рублей. Сколько будет стоить товар со скидкой?",
    "У вас есть 3-литровое ведро и 5-литровое ведро. Как с их помощью отмерить ровно 4 литра воды?"
]

system_prompt = "Ты логический математический счётчик. Твоя задача перед тем как дать ответ написать каждый шаг и каждую операцию что ты выполнил . В формате Шаг 1: ... Шаг 2:... ... Ответ:... Каждое действие должно быть описано и использоватся в контексте решения задачи ."  # настрой системный промпт для CoT

# Проверка
for i, task in enumerate(tasks, 1):
    answer = generate_answer(system_prompt=system_prompt, user_prompt=task)
    print(f"\n--- Задача {i} ---")
    print(f"Вопрос: {task}")
    print(f"Ответ модели:\n{answer}")
    print("-" * 40)


--- Задача 1 ---
Вопрос: У Маши было 5 яблок. Она отдала 2 яблока Пете, а потом нашла ещё 3. Сколько яблок у Маши теперь?
Ответ модели:
Шаг 1: Определяем начальное количество яблок у Маши.
Маша изначально имела 5 яблок.

Шаг 2: Учитываем, что Маша отдала 2 яблока Пете.
Чтобы найти, сколько яблок осталось у Маши после отдачи, вычитаем количество отданных яблок из начального количества.
5 - 2 = 3 яблока

Шаг 3: Учитываем, что Маша нашла ещё 3 яблока.
После того, как Маша нашла яблоки, нужно добавить это число к оставшемуся количеству яблок.
3 + 3 = 6 яблок

Шаг 4: Записываем окончательный ответ.
У Маши теперь 6 яблок.

Ответ: 6
----------------------------------------

--- Задача 2 ---
Вопрос: Если все люди смертны, а Сократ — человек, то смертен ли Сократ?
Ответ модели:
Давайте решим эту задачу, используя логику и следуя вашему формату.

**Шаг 1: Анализ посылок.**
У нас есть две посылки, которые мы должны интерпретировать как логические утверждения:
    *  Посылка 1: Все люди смертны. 